# Alanine dipeptide: SBG versus full-covariance GMM + Proposition 3

This notebook compares the alanine-dipeptide experiment from *Scalable Equilibrium Sampling with Sequential Boltzmann Generators* against our Method 2 and first diagnoses the mode structure of the biased training trajectory.

- **Paper baseline (SBG):** pretrained TarFlow proposal followed by the repository's continuous-time ULA-SMC implementation.
- **Method 2:** infer physical modes from heavy-atom-distance PCA/TICA, fit one full-covariance Cartesian Gaussian at each selected mode location, train one shared ECNF++ EGNN velocity using only within-mode flow pairs, and use that velocity in a Proposition-3 annealed sampler from the marginal GMM potential to the exact OpenMM target.

The target, trajectory split, force field, normalization, test reference, evaluator, particle count, annealing schedule, and resampling threshold come from the official Ace-A-Nme configuration. The GMM uses full covariance matrices; `reg_covar` is only a Cholesky-stability ridge. Cartesian configurations are centered and represented in the 63-dimensional center-of-mass-free subspace. Random rotation augmentation is deliberately not used because a fixed Cartesian GMM is not rotationally invariant.

Set `SMOKE_TEST=False` for the paper-scale comparison.


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 
!echo $CUDA_VISIBLE_DEVICES

0


In [3]:
from __future__ import annotations

import math
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import mdtraj as md
import numpy as np
import torch
import torch.nn as nn
from dotenv import load_dotenv
from sklearn.mixture import GaussianMixture

load_dotenv(override=True)
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src" / "transferable_samplers").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("Run this notebook from the transferable-samplers-prop3 repository.")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from transferable_samplers.data.single_peptide_datamodule import SinglePeptideDataModule
from transferable_samplers.evaluation.evaluator import PeptideEnsembleEvaluator
from transferable_samplers.nn.egnn.egnn_dynamics_ad2_cat import EGNN_dynamics_AD2_cat
from transferable_samplers.utils.dataclasses import SamplesData
from transferable_samplers.utils.standardization import standardize_coords

SEED = 4201
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
SMOKE_TEST = True
RUN_OFFICIAL_SBG = False

MODE_COUNT_OVERRIDE = None  # Set an integer after inspection if the diagnostics disagree.
GMM_REG_COVAR = 1e-4
TRAIN_STEPS = 500 if SMOKE_TEST else 100_000
TRAIN_BATCH = 128 if SMOKE_TEST else 512
NUM_PARTICLES = 256 if SMOKE_TEST else 10_000
NUM_ANNEALING_STEPS = 20 if SMOKE_TEST else 100
EPSILON = 1e-5
ESS_THRESHOLD = 0.5
HUTCHINSON_SAMPLES = 1 if SMOKE_TEST else 4
DRIFT_BATCH = 32 if SMOKE_TEST else 128
ENERGY_BATCH = 64 if SMOKE_TEST else 256
LEARNING_RATE = 5e-4
SIGMA_MIN = 0.0

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_float32_matmul_precision("highest")
print({
    "root": str(REPO_ROOT),
    "device": str(DEVICE),
    "smoke_test": SMOKE_TEST,
    "hutchinson_samples": HUTCHINSON_SAMPLES,
})


ModuleNotFoundError: No module named 'dotenv'

## 1. Official SBG baseline

The command below runs the repository's unmodified `tarflow_Ace-A-Nme_ula` experiment. Its paper-scale defaults are 10,000 particles, 100 annealing steps, `init_eps=1e-5`, ESS threshold 0.5, multinomial resampling, target-energy cutoff 10, and 0.2% log-weight filtering. Results are saved separately and reused by the comparison cells.


In [ ]:
if "SCRATCH_DIR" not in os.environ:
    default_scratch = REPO_ROOT / "scratch"
    os.environ["SCRATCH_DIR"] = str(default_scratch)
    print(f"SCRATCH_DIR was not set; using ignored local cache: {default_scratch}")

ARTIFACT_ROOT = REPO_ROOT / "outputs" / "aldp_sbg_vs_prop3"
SBG_OUTPUT = ARTIFACT_ROOT / "official_sbg"
SBG_SAMPLE_FILE = SBG_OUTPUT / "test" / "Ace-A-Nme" / "samples_dict.pt"
SBG_DIAGNOSTICS_FILE = SBG_OUTPUT / "test" / "Ace-A-Nme" / "diagnostics.pt"

baseline_command = [
    "uv", "run", "python", "-m", "transferable_samplers.eval",
    "experiment=single_system/eval/tarflow_Ace-A-Nme_ula",
    "trainer=gpu" if torch.cuda.is_available() else "trainer=cpu",
    "logger=csv",
    f"paths.output_dir={SBG_OUTPUT}",
    f"callbacks.sampling_evaluation.output_dir={SBG_OUTPUT}",
]
print("Official SBG command:\n", " ".join(map(str, baseline_command)))
if RUN_OFFICIAL_SBG:
    SBG_OUTPUT.mkdir(parents=True, exist_ok=True)
    subprocess.run(baseline_command, cwd=REPO_ROOT, check=True)
elif not SBG_SAMPLE_FILE.exists():
    print("SBG artifact is absent. Set RUN_OFFICIAL_SBG=True to create it.")


## 2. Reuse the paper trajectory and OpenMM target

Ace-A-Nme is the paper's 300 K alanine-dipeptide system with 22 atoms. The datamodule downloads the official train/test splits and constructs the same OpenMM target and test reference used by the evaluator.


In [ ]:
scratch_root = Path(os.environ["SCRATCH_DIR"]) / "transferable-samplers"
datamodule = SinglePeptideDataModule(
    data_dir=str(scratch_root / "sequential-boltzmann-generators-data"),
    sequence="Ace-A-Nme",
    temperature=300,  # Keep this integer: the Hugging Face directory is Ace-A-Nme_300K.
    num_dimensions=3,
    num_atoms=22,
    batch_size=512,
    num_workers=0,
    num_eval_samples=10_000,
)
datamodule.prepare_data()
required_dataset_files = [
    datamodule.train_data_path,
    datamodule.val_data_path,
    datamodule.test_data_path,
    datamodule.pdb_path,
]
missing_dataset_files = [path for path in required_dataset_files if not Path(path).exists()]
if missing_dataset_files:
    raise FileNotFoundError(
        "Dataset download completed without the expected Ace-A-Nme_300K files:\n"
        + "\n".join(missing_dataset_files)
    )
train_raw = torch.from_numpy(np.load(datamodule.train_data_path)).float()
datamodule.std = (train_raw - train_raw.mean(dim=1, keepdim=True)).std()
eval_context = datamodule.prepare_eval(sequence="Ace-A-Nme", stage="test")
train_x = standardize_coords(train_raw, datamodule.std).to(DEVICE)

num_atoms, spatial_dim = train_x.shape[1:]
projector = torch.eye(num_atoms, dtype=torch.float64) - torch.ones(num_atoms, num_atoms, dtype=torch.float64) / num_atoms
eigenvalues, eigenvectors = torch.linalg.eigh(projector)
Q = torch.kron(eigenvectors[:, eigenvalues > 0.5], torch.eye(spatial_dim, dtype=torch.float64)).to(DEVICE, DTYPE)
DIM = Q.shape[1]

def x_to_y(x: torch.Tensor) -> torch.Tensor:
    return x.reshape(len(x), -1) @ Q

def y_to_x(y: torch.Tensor) -> torch.Tensor:
    return (y @ Q.T).reshape(len(y), num_atoms, spatial_dim)

train_y = x_to_y(train_x)
print({
    "train_samples": len(train_y),
    "mean_free_dimension": DIM,
    "normalization_std": float(datamodule.std),
    "test_reference": len(eval_context.true_data),
})


## 3. Discover the biased-data mode count

The centered Cartesian data live in a 63-dimensional subspace of the nominal 66 coordinates because translation has been removed. A Cartesian density mode can still be caused by orientation or alignment, so this diagnostic compares it with rotation- and translation-invariant heavy-atom distances; hydrogens are excluded so methyl rotation or equivalent-hydrogen labeling is not mistaken for a conformational mode. PCA gives a density view, while TICA uses the trajectory ordering to expose slow/metastable structure. For each representation, full-covariance GMMs are scanned over candidate counts and compared using BIC, block-held-out log likelihood, silhouette separation, and agreement between independent random initializations. TICA timescale gaps provide a separate kinetic estimate.

The training count is selected only when at least two of the internal-distance PCA BIC, internal-distance TICA BIC, and TICA timescale-gap estimates agree. Otherwise, inspect the plots, set `MODE_COUNT_OVERRIDE`, and rerun this cell. A GMM may use several components for one curved basin, so the diagnostics and Ramachandran projection should still be checked before accepting the result.


In [ ]:
import deeptime as dt
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

MODE_ANALYSIS_SAMPLES = 5_000 if SMOKE_TEST else 30_000
MODE_PCA_DIM = 15
MODE_TICA_DIM = 5
MODE_TICA_LAG = 25 if SMOKE_TEST else 100  # lag in retained trajectory frames
MODE_CANDIDATES = np.arange(1, 11)
MODE_GMM_N_INIT = 1 if SMOKE_TEST else 3
MODE_SILHOUETTE_SAMPLES = 2_000 if SMOKE_TEST else 5_000

mode_count = min(MODE_ANALYSIS_SAMPLES, len(train_x))
mode_indices = np.linspace(0, len(train_x) - 1, mode_count, dtype=int)
mode_x = train_x[torch.as_tensor(mode_indices, device=DEVICE)].detach().cpu()
mode_y = train_y[torch.as_tensor(mode_indices, device=DEVICE)].detach().cpu().numpy()
mode_xyz = mode_x * datamodule.std.cpu()

heavy_atom_indices = torch.as_tensor([
    atom.index for atom in eval_context.topology.atoms
    if atom.element is not None and atom.element.symbol != "H"
])
heavy_pair_i, heavy_pair_j = torch.triu_indices(
    len(heavy_atom_indices), len(heavy_atom_indices), offset=1
)
pair_i = heavy_atom_indices[heavy_pair_i]
pair_j = heavy_atom_indices[heavy_pair_j]
distance_parts = []
for start in range(0, mode_count, 5_000):
    xyz_batch = mode_xyz[start : start + 5_000]
    distance_parts.append(torch.linalg.vector_norm(
        xyz_batch[:, pair_i] - xyz_batch[:, pair_j], dim=-1
    ))
mode_distances = torch.cat(distance_parts).numpy()

cartesian_scaler = StandardScaler().fit(mode_y)
distance_scaler = StandardScaler().fit(mode_distances)
cartesian_scaled = cartesian_scaler.transform(mode_y)
distance_scaled = distance_scaler.transform(mode_distances)
cartesian_pca = PCA(
    n_components=min(MODE_PCA_DIM, cartesian_scaled.shape[1]), random_state=SEED
).fit(cartesian_scaled)
distance_pca = PCA(
    n_components=min(MODE_PCA_DIM, distance_scaled.shape[1]), random_state=SEED
).fit(distance_scaled)
cartesian_view = cartesian_pca.transform(cartesian_scaled)
distance_view = distance_pca.transform(distance_scaled)

tica_lag = min(MODE_TICA_LAG, max(1, mode_count // 10))
tica_model_biased = dt.decomposition.TICA(
    dim=min(MODE_TICA_DIM, distance_scaled.shape[1]), lagtime=tica_lag
).fit(distance_scaled).fetch_model()
tica_view = np.asarray(tica_model_biased.transform(distance_scaled))
tica_timescales = np.asarray(
    tica_model_biased.timescales(), dtype=float
)[: tica_view.shape[1]]
finite_tica_timescales = tica_timescales[
    np.isfinite(tica_timescales) & (tica_timescales > 0)
]
if len(finite_tica_timescales) >= 2:
    timescale_gap_ratios = finite_tica_timescales[:-1] / finite_tica_timescales[1:]
    tica_gap_mode_count = int(np.argmax(timescale_gap_ratios) + 2)
else:
    timescale_gap_ratios = np.asarray([])
    tica_gap_mode_count = None

mode_views = {
    "centered Cartesian PCA": cartesian_view,
    "internal-distance PCA": distance_view,
    "internal-distance TICA": tica_view,
}

def scan_mode_counts(features: np.ndarray, seed: int):
    block_length = max(5 * tica_lag, 50)
    block_id = np.arange(len(features)) // block_length
    validation_mask = block_id % 5 == 4
    if validation_mask.sum() < 2:
        validation_mask[-max(2, len(features) // 5):] = True
    fit_features = features[~validation_mask]
    validation_features = features[validation_mask]
    if len(validation_features) < 2:
        raise ValueError("Mode analysis needs at least ten biased samples.")

    records, fitted_models = [], {}
    for candidate in MODE_CANDIDATES:
        primary = GaussianMixture(
            n_components=int(candidate), covariance_type="full",
            reg_covar=1e-5, n_init=MODE_GMM_N_INIT, max_iter=500,
            random_state=seed + int(candidate),
        ).fit(fit_features)
        alternate = GaussianMixture(
            n_components=int(candidate), covariance_type="full",
            reg_covar=1e-5, n_init=1, max_iter=500,
            random_state=seed + 10_000 + int(candidate),
        ).fit(fit_features)
        validation_labels = primary.predict(validation_features)
        alternate_labels = alternate.predict(validation_features)
        if candidate == 1 or len(np.unique(validation_labels)) < 2:
            silhouette = np.nan
        else:
            silhouette = silhouette_score(
                validation_features, validation_labels,
                sample_size=min(MODE_SILHOUETTE_SAMPLES, len(validation_features)),
                random_state=seed,
            )
        records.append({
            "k": int(candidate),
            "bic": float(primary.bic(fit_features)),
            "heldout_log_likelihood": float(primary.score(validation_features)),
            "silhouette": float(silhouette),
            "seed_stability_ari": float(adjusted_rand_score(
                validation_labels, alternate_labels
            )),
        })
        fitted_models[int(candidate)] = primary
    return records, fitted_models

mode_scan, mode_fits = {}, {}
for view_index, (view_name, view_features) in enumerate(mode_views.items()):
    mode_scan[view_name], mode_fits[view_name] = scan_mode_counts(
        view_features, SEED + 100 * (view_index + 1)
    )

mode_summary = {}
print("view | BIC choice | held-out choice | silhouette choice | ARI at BIC choice")
for view_name, records in mode_scan.items():
    best_bic = min(records, key=lambda record: record["bic"])["k"]
    best_heldout = max(records, key=lambda record: record["heldout_log_likelihood"])["k"]
    finite_silhouette = [record for record in records if np.isfinite(record["silhouette"])]
    best_silhouette = max(finite_silhouette, key=lambda record: record["silhouette"])["k"]
    bic_record = next(record for record in records if record["k"] == best_bic)
    mode_summary[view_name] = {
        "bic": best_bic, "heldout": best_heldout,
        "silhouette": best_silhouette,
        "bic_seed_stability_ari": bic_record["seed_stability_ari"],
    }
    print(
        f"{view_name} | {best_bic:10d} | {best_heldout:15d} | "
        f"{best_silhouette:17d} | {bic_record['seed_stability_ari']:.3f}"
    )
    if best_bic == int(MODE_CANDIDATES[-1]) or best_heldout == int(MODE_CANDIDATES[-1]):
        print(f"  warning: {view_name} selected the scan boundary; increase MODE_CANDIDATES.")
print({
    "analysis_samples": mode_count,
    "heavy_atoms": len(heavy_atom_indices),
    "internal_distance_features": mode_distances.shape[1],
    "TICA_lag_in_retained_frames": tica_lag,
    "TICA_timescales": np.round(finite_tica_timescales, 3).tolist(),
    "TICA_timescale_gap_mode_estimate": tica_gap_mode_count,
    "Cartesian_PCA_explained_fraction": float(cartesian_pca.explained_variance_ratio_.sum()),
    "distance_PCA_explained_fraction": float(distance_pca.explained_variance_ratio_.sum()),
})

fig, axes = plt.subplots(3, 3, figsize=(14, 11), constrained_layout=True)
for column, (view_name, records) in enumerate(mode_scan.items()):
    candidates = np.asarray([record["k"] for record in records])
    bic = np.asarray([record["bic"] for record in records])
    heldout = np.asarray([record["heldout_log_likelihood"] for record in records])
    silhouette = np.asarray([record["silhouette"] for record in records])
    stability = np.asarray([record["seed_stability_ari"] for record in records])
    axes[0, column].plot(candidates, bic - bic.min(), marker="o")
    axes[0, column].set(title=view_name, ylabel=r"$\Delta$BIC (lower is better)")
    axes[1, column].plot(candidates, heldout, marker="o")
    axes[1, column].set(ylabel="held-out log likelihood / sample")
    axes[2, column].plot(candidates, silhouette, marker="o", label="silhouette")
    axes[2, column].plot(candidates, stability, marker="s", label="seed ARI")
    axes[2, column].set(xlabel="candidate mixture components K", ylabel="score", ylim=(-0.1, 1.05))
    axes[2, column].legend()
plt.show()

if len(finite_tica_timescales):
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)
    axes[0].plot(
        np.arange(1, len(finite_tica_timescales) + 1),
        finite_tica_timescales, marker="o",
    )
    axes[0].set(
        xlabel="nontrivial TICA process", ylabel="implied timescale",
        title="Biased-trajectory TICA timescales",
    )
    if len(timescale_gap_ratios):
        axes[1].plot(
            np.arange(2, len(timescale_gap_ratios) + 2),
            timescale_gap_ratios, marker="o",
        )
    axes[1].set(
        xlabel="states implied by gap position", ylabel="adjacent timescale ratio",
        title=f"Largest-gap estimate: {tica_gap_mode_count} modes",
    )
    plt.show()

scatter_count = min(10_000, mode_count)
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for ax, (view_name, view_features) in zip(axes, mode_views.items()):
    selected_k = mode_summary[view_name]["bic"]
    labels = mode_fits[view_name][selected_k].predict(view_features[:scatter_count])
    ax.scatter(
        view_features[:scatter_count, 0], view_features[:scatter_count, 1],
        c=labels, cmap="tab20", s=3, alpha=0.35, rasterized=True,
    )
    ax.set(
        xlabel="coordinate 1", ylabel="coordinate 2",
        title=f"{view_name} (BIC K={selected_k})",
    )
plt.show()

internal_choices = {
    view_name: (summary["bic"], summary["heldout"])
    for view_name, summary in mode_summary.items()
    if view_name != "centered Cartesian PCA"
}
print("Internal-coordinate BIC/held-out choices:", internal_choices)
mode_count_votes = [
    mode_summary["internal-distance PCA"]["bic"],
    mode_summary["internal-distance TICA"]["bic"],
    tica_gap_mode_count,
]
mode_count_votes = [vote for vote in mode_count_votes if vote is not None]
vote_values, vote_counts = np.unique(mode_count_votes, return_counts=True)
consensus_mode_count = (
    int(vote_values[np.argmax(vote_counts)])
    if len(vote_counts) and vote_counts.max() >= 2
    else None
)
consensus_hits_scan_boundary = (
    consensus_mode_count == int(MODE_CANDIDATES[-1])
)
SELECTED_MODE_COUNT = (
    int(MODE_COUNT_OVERRIDE)
    if MODE_COUNT_OVERRIDE is not None
    else None if consensus_hits_scan_boundary else consensus_mode_count
)
print({
    "mode_count_votes_(distance_PCA_BIC,TICA_BIC,TICA_gap)": mode_count_votes,
    "consensus_mode_count": consensus_mode_count,
    "consensus_hits_scan_boundary": consensus_hits_scan_boundary,
    "override": MODE_COUNT_OVERRIDE,
    "selected_mode_count_for_training": SELECTED_MODE_COUNT,
})
print(
    "Treat a mode count as supported only when the internal PCA and TICA choices "
    "agree or form a stable plateau with high ARI; a Cartesian-only choice can be an alignment artifact."
)


## 4. Fit the mode-informed full-covariance GMM prior

A final GMM in heavy-atom-distance TICA space assigns every biased conformation to one of the selected modes. Those labels are transferred back to the centered 63-dimensional Cartesian representation. One unrestricted full-covariance Gaussian is fitted at each mode location, and its mixture weight is the empirical biased-data mode frequency. The TICA centers classify modes; the Cartesian means and covariances define the actual marginal GMM prior used by the shared drift and Proposition 3.


In [ ]:
if SELECTED_MODE_COUNT is None:
    raise RuntimeError(
        "The internal-coordinate diagnostics did not produce a usable interior "
        "two-vote mode-count consensus. Inspect the plots, set MODE_COUNT_OVERRIDE "
        "in the configuration "
        "cell, and rerun from the mode analysis."
    )
NUM_COMPONENTS = int(SELECTED_MODE_COUNT)
if NUM_COMPONENTS < 1:
    raise ValueError("SELECTED_MODE_COUNT must be positive.")

mode_classifier = GaussianMixture(
    n_components=NUM_COMPONENTS,
    covariance_type="full",
    reg_covar=1e-5,
    n_init=5,
    max_iter=500,
    random_state=SEED + 500,
).fit(tica_view)
mode_order = np.argsort(mode_classifier.means_[:, 0])
mode_label_remap = np.empty(NUM_COMPONENTS, dtype=int)
mode_label_remap[mode_order] = np.arange(NUM_COMPONENTS)
mode_tica_centers = mode_classifier.means_[mode_order]

def classify_modes(samples: torch.Tensor, batch_size: int = 20_000):
    labels = []
    for start in range(0, len(samples), batch_size):
        coordinates = samples[start : start + batch_size].detach().cpu()
        coordinates = coordinates * datamodule.std.cpu()
        distances = torch.linalg.vector_norm(
            coordinates[:, pair_i] - coordinates[:, pair_j], dim=-1
        ).numpy()
        scaled_distances = distance_scaler.transform(distances)
        tica_coordinates = np.asarray(
            tica_model_biased.transform(scaled_distances)
        )
        raw_labels = mode_classifier.predict(tica_coordinates)
        labels.append(torch.as_tensor(
            mode_label_remap[raw_labels], dtype=torch.long
        ))
    return torch.cat(labels).to(DEVICE)

train_component = classify_modes(train_x)
component_pools = [
    torch.where(train_component == component)[0]
    for component in range(NUM_COMPONENTS)
]
if any(len(pool) < 2 for pool in component_pools):
    raise RuntimeError("Every selected mode needs at least two biased training samples.")

component_counts = torch.bincount(train_component, minlength=NUM_COMPONENTS)
gmm_weights = component_counts.to(device=DEVICE, dtype=DTYPE)
gmm_weights = gmm_weights / gmm_weights.sum()

def empirical_mode_gaussian(pool: torch.Tensor, batch_size: int = 20_000):
    count = len(pool)
    coordinate_sum = torch.zeros(DIM, device=DEVICE, dtype=torch.float64)
    for start in range(0, count, batch_size):
        values = train_y[pool[start : start + batch_size]].to(torch.float64)
        coordinate_sum += values.sum(dim=0)
    mean = coordinate_sum / count

    scatter = torch.zeros(DIM, DIM, device=DEVICE, dtype=torch.float64)
    for start in range(0, count, batch_size):
        values = train_y[pool[start : start + batch_size]].to(torch.float64)
        centered = values - mean
        scatter += centered.T @ centered
    covariance = scatter / count
    covariance += GMM_REG_COVAR * torch.eye(
        DIM, device=DEVICE, dtype=torch.float64
    )
    return mean.to(DTYPE), covariance.to(DTYPE)

mode_gaussians = [
    empirical_mode_gaussian(pool) for pool in component_pools
]
gmm_means = torch.stack([mean for mean, _ in mode_gaussians])
gmm_covariances = torch.stack([
    covariance for _, covariance in mode_gaussians
])
gmm_cholesky = torch.linalg.cholesky(gmm_covariances)
gmm_precision = torch.cholesky_inverse(gmm_cholesky)
gmm_logdet = 2 * torch.log(
    torch.diagonal(gmm_cholesky, dim1=-2, dim2=-1)
).sum(-1)

def sample_gmm(
    n: int,
    generator: torch.Generator | None = None,
    return_component: bool = False,
):
    component = torch.multinomial(
        gmm_weights, n, replacement=True, generator=generator
    )
    noise = torch.randn(
        n, DIM, device=DEVICE, dtype=DTYPE, generator=generator
    )
    y = gmm_means[component] + torch.bmm(
        gmm_cholesky[component], noise.unsqueeze(-1)
    ).squeeze(-1)
    return (y, component) if return_component else y

def gmm_energy_score(y: torch.Tensor):
    delta = y[:, None, :] - gmm_means[None, :, :]
    precision_delta = torch.einsum(
        "kij,bkj->bki", gmm_precision, delta
    )
    mahalanobis = torch.sum(delta * precision_delta, dim=-1)
    component_logp = (
        torch.log(gmm_weights)[None, :]
        - 0.5 * (
            DIM * math.log(2 * math.pi)
            + gmm_logdet[None, :]
            + mahalanobis
        )
    )
    responsibility = torch.softmax(component_logp, dim=1)
    score = torch.sum(
        responsibility[..., None] * (-precision_delta), dim=1
    )
    return -torch.logsumexp(component_logp, dim=1), score

print("selected mode count:", NUM_COMPONENTS)
print("empirical mode weights:", np.round(gmm_weights.cpu().numpy(), 5))
print("assigned counts:", component_counts.cpu().numpy())
print("mode locations in TICA space:\n", np.round(mode_tica_centers, 4))
print(
    "Cartesian covariance condition numbers:",
    np.round(torch.linalg.cond(gmm_covariances).cpu().numpy(), 2),
)


### Initial mode-informed GMM Ramachandran density

This is the unweighted prior population before shared-drift training or Proposition-3 transport. It uses the same seed as `run_prop3`, so these are exactly the particles that initialize the later Method-2 trajectory. Each component is one data-derived mode rather than an arbitrary Gaussian subdivision.


In [ ]:
def rama(samples: torch.Tensor, normalized: bool = True):
    xyz = samples.detach().cpu()
    if normalized:
        xyz = xyz * datamodule.std.cpu()
    xyz = xyz.numpy()
    trajectory = md.Trajectory(xyz, eval_context.topology)
    phi = md.compute_phi(trajectory)[1].reshape(len(samples), -1)[:, 0]
    psi = md.compute_psi(trajectory)[1].reshape(len(samples), -1)[:, 0]
    return np.rad2deg(phi), np.rad2deg(psi)

initial_gmm_generator = torch.Generator(device=DEVICE).manual_seed(SEED + 2)
initial_gmm_y, initial_gmm_component = sample_gmm(
    NUM_PARTICLES, generator=initial_gmm_generator, return_component=True
)
initial_gmm_samples = y_to_x(initial_gmm_y).detach()
initial_gmm_phi, initial_gmm_psi = rama(initial_gmm_samples)
initial_component_mass = (
    torch.bincount(initial_gmm_component, minlength=NUM_COMPONENTS).float()
    / len(initial_gmm_component)
)
print("initial sampled mode fractions:", initial_component_mass.cpu().numpy())

fig, ax = plt.subplots(figsize=(5.2, 4.4), constrained_layout=True)
density = ax.hexbin(
    initial_gmm_phi, initial_gmm_psi, gridsize=55, bins="log", mincnt=1
)
fig.colorbar(density, ax=ax, label="log count")
ax.set(
    xlim=(-180, 180), ylim=(-180, 180),
    xlabel=r"$\phi$ (deg)", ylabel=r"$\psi$ (deg)",
    title=f"Initial mode-informed GMM prior (K={NUM_COMPONENTS})",
)
plt.show()


### Initial mode-informed GMM energy and distance comparison

The actual mode-informed prior particles are evaluated under the exact OpenMM target and compared with an equally sized subset of test MD conformations. Interatomic distances use the evaluator's definition: all unique upper-triangular atom-pair distances pooled across conformations. Shared histogram bins and the energy overflow bin match the repository's plotting convention.


In [ ]:
from scipy.stats import wasserstein_distance

@torch.no_grad()
def batched_target_energy(samples: torch.Tensor, batch_size: int = ENERGY_BATCH):
    energies = []
    for start in range(0, len(samples), batch_size):
        energy = eval_context.target_energy.energy(
            samples[start : start + batch_size]
        )
        energies.append(energy.detach().cpu())
    return torch.cat(energies)

def pooled_interatomic_distances(samples: torch.Tensor):
    samples = samples.detach().cpu()
    num_atoms = samples.shape[1]
    upper = torch.triu(
        torch.ones(num_atoms, num_atoms, dtype=torch.bool), diagonal=1
    )
    return torch.cdist(samples, samples)[:, upper].reshape(-1)

comparison_count = min(len(initial_gmm_samples), len(eval_context.true_data.samples))
gmm_compare = initial_gmm_samples[:comparison_count]
md_compare = eval_context.true_data.samples[:comparison_count].detach().cpu()
gmm_prior_energy = batched_target_energy(gmm_compare)
md_energy = eval_context.true_data.E_target[:comparison_count].detach().cpu()
gmm_physical = gmm_compare.detach().cpu() * datamodule.std.cpu()
gmm_distances = pooled_interatomic_distances(gmm_physical)
md_distances = pooled_interatomic_distances(md_compare)

gmm_energy_finite = torch.isfinite(gmm_prior_energy)
md_energy_finite = torch.isfinite(md_energy)
if not bool(gmm_energy_finite.all() and md_energy_finite.all()):
    print({
        "finite_GMM_energy_fraction": float(gmm_energy_finite.float().mean()),
        "finite_MD_energy_fraction": float(md_energy_finite.float().mean()),
    })
gmm_energy_for_metrics = gmm_prior_energy[gmm_energy_finite]
md_energy_for_metrics = md_energy[md_energy_finite]

gmm_prior_comparison = {
    "num_conformations": comparison_count,
    "GMM_mean_target_energy": float(gmm_energy_for_metrics.mean()),
    "MD_mean_target_energy": float(md_energy_for_metrics.mean()),
    "GMM_median_target_energy": float(gmm_energy_for_metrics.median()),
    "MD_median_target_energy": float(md_energy_for_metrics.median()),
    "energy_W1": wasserstein_distance(
        md_energy_for_metrics.numpy(), gmm_energy_for_metrics.numpy()
    ),
    "GMM_fraction_energy_at_least_100": float((gmm_energy_for_metrics >= 100).float().mean()),
    "MD_fraction_energy_at_least_100": float((md_energy_for_metrics >= 100).float().mean()),
    "pooled_interatomic_distance_W1": wasserstein_distance(
        md_distances.numpy(), gmm_distances.numpy()
    ),
}
print(gmm_prior_comparison)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
energy_cap = 100.0
energy_min = float(torch.minimum(
    md_energy_for_metrics.min(), gmm_energy_for_metrics.min()
))
energy_bins = np.linspace(energy_min, energy_cap, 100)
for energy, label, color in [
    (md_energy_for_metrics, "test MD", "tab:green"),
    (gmm_energy_for_metrics, "fitted GMM prior", "tab:orange"),
]:
    axes[0].hist(
        torch.clamp(energy, max=energy_cap - 1e-4).numpy(),
        bins=energy_bins, density=True, histtype="step", lw=2, label=label, color=color
    )
axes[0].set(
    xlabel="exact target energy (last bin: >=100)", ylabel="density",
    title=f"Target energy; W1={gmm_prior_comparison['energy_W1']:.3g}"
)
axes[0].legend()

distance_min = float(torch.minimum(md_distances.min(), gmm_distances.min()))
distance_max = float(torch.maximum(md_distances.max(), gmm_distances.max()))
distance_bins = np.linspace(distance_min, distance_max, 100)
axes[1].hist(
    md_distances.numpy(), bins=distance_bins, density=True, histtype="step",
    lw=2, label="test MD", color="tab:green"
)
axes[1].hist(
    gmm_distances.numpy(), bins=distance_bins, density=True, histtype="step",
    lw=2, label="fitted GMM prior", color="tab:orange"
)
axes[1].set(
    xlabel="interatomic distance (nm)", ylabel="density",
    title=(
        "Pooled atom-pair distances; "
        f"W1={gmm_prior_comparison['pooled_interatomic_distance_W1']:.3g}"
    ),
)
axes[1].legend()
plt.show()


### Project the selected high-dimensional modes into Ramachandran space

This validation plot projects the final heavy-atom-distance TICA labels assigned to the biased training subset into $(\phi,\psi)$ space. It shows whether the high-dimensional modes correspond to recognizable conformational regions and reports the empirical weights that define the mode-informed prior.


In [ ]:
mode_trajectory = md.Trajectory(mode_xyz.numpy(), eval_context.topology)
mode_phi = md.compute_phi(mode_trajectory)[1].reshape(mode_count, -1)[:, 0]
mode_psi = md.compute_psi(mode_trajectory)[1].reshape(mode_count, -1)[:, 0]
mode_index_tensor = torch.as_tensor(mode_indices, device=DEVICE)
rama_mode_labels = train_component[mode_index_tensor].detach().cpu().numpy()
rama_mode_weights = gmm_weights.detach().cpu().numpy()
print("selected mode weights:", np.round(rama_mode_weights, 5))

rama_plot_count = min(30_000, mode_count)
rama_plot_indices = np.linspace(0, mode_count - 1, rama_plot_count, dtype=int)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)
scatter = axes[0].scatter(
    np.rad2deg(mode_phi[rama_plot_indices]),
    np.rad2deg(mode_psi[rama_plot_indices]),
    c=rama_mode_labels[rama_plot_indices],
    cmap="tab20",
    vmin=-0.5,
    vmax=NUM_COMPONENTS - 0.5,
    s=3,
    alpha=0.25,
    rasterized=True,
)
axes[0].set(
    xlim=(-180, 180), ylim=(-180, 180),
    xlabel=r"$\phi$ (deg)", ylabel=r"$\psi$ (deg)",
    title=f"Selected high-dimensional modes (K={NUM_COMPONENTS})",
)
fig.colorbar(scatter, ax=axes[0], label="mode label")
axes[1].bar(
    np.arange(NUM_COMPONENTS), rama_mode_weights, color="tab:blue"
)
axes[1].set(
    xticks=np.arange(NUM_COMPONENTS),
    xlabel="mode",
    ylabel="biased-data fraction",
    ylim=(0, max(0.55, 1.1 * rama_mode_weights.max())),
    title="Empirical mode weights used by the prior",
)
plt.show()


## 5. Train one shared ECNF++ drift with within-mode pairs

This uses one copy of the paper repository's equivariant EGNN architecture: width 256 and depth 5. The network is shared across all modes and acts in 66 Cartesian coordinates, while the wrapper projects its mean-free velocity into the 63-dimensional GMM coordinates. For each source sampled from mode $z$, the target is drawn only from the biased-data pool labelled $z$. Thus there are no cross-mode flow-matching pairs, but there are also no separate mode-specific networks.


In [ ]:
class ProjectedEGNNVelocity(nn.Module):
    def __init__(self):
        super().__init__()
        self.egnn = EGNN_dynamics_AD2_cat(
            num_atoms=num_atoms,
            num_dimensions=spatial_dim,
            channels=256,
            num_layers=5,
        )
        self.register_buffer("basis", Q)

    def forward(self, y: torch.Tensor, t: torch.Tensor):
        x_flat = y @ self.basis.T
        velocity_x = self.egnn(t, x_flat)
        return velocity_x @ self.basis

drift = ProjectedEGNNVelocity().to(DEVICE)
optimizer = torch.optim.AdamW(drift.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
generator = torch.Generator(device=DEVICE).manual_seed(SEED + 1)
loss_history = []
drift.train()

for step in range(TRAIN_STEPS):
    y0, component = sample_gmm(TRAIN_BATCH, generator=generator, return_component=True)
    y1 = torch.empty_like(y0)
    for component_index, pool in enumerate(component_pools):
        mask = component == component_index
        count = int(mask.sum())
        if count:
            chosen = pool[torch.randint(len(pool), (count,), device=DEVICE, generator=generator)]
            y1[mask] = train_y[chosen]

    t = torch.rand(TRAIN_BATCH, 1, device=DEVICE, generator=generator)
    yt = (1 - (1 - SIGMA_MIN) * t) * y0 + t * y1
    target_velocity = y1 - (1 - SIGMA_MIN) * y0
    loss = (drift(yt, t[:, 0]) - target_velocity).square().mean()

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(drift.parameters(), 10.0)
    optimizer.step()
    loss_history.append(float(loss.detach()))
    if (step + 1) % max(100, TRAIN_STEPS // 10) == 0:
        print(f"{step + 1}/{TRAIN_STEPS}: loss={loss_history[-1]:.6f}")

drift.eval()
plt.figure(figsize=(6, 3))
plt.plot(loss_history)
plt.yscale("log")
plt.xlabel("training step")
plt.ylabel("flow-matching MSE")
plt.title("One shared EGNN trained with within-mode pairs")
plt.show()


## 6. Proposition-3 SMC

For (U_t=(1-t)U_0+tU_1), the dynamics and incremental log weight are

[
dX_t = [-\epsilon\nabla U_t(X_t)+v_t(X_t)]dt + \sqrt{2\epsilon},dW_t,
]

[
d\log w_t = [\nabla\!\cdot v_t-\nabla U_t\!\cdot v_t+U_0-U_1]dt.
]

Proposition 3 uses the marginal mode-informed GMM energy, not a mode-conditioned energy. The learned velocity is the same shared drift for every particle. The initial mode index is retained only as ancestry metadata and is resampled with its particle for diagnostics. The divergence is estimated with Hutchinson probes, and multinomial resampling uses the same ESS threshold as the SBG configuration.


In [ ]:
@torch.no_grad()
def target_energy_score(y: torch.Tensor):
    energies, scores = [], []
    for start in range(0, len(y), ENERGY_BATCH):
        y_batch = y[start : start + ENERGY_BATCH]
        x_batch = y_to_x(y_batch)
        energy, gradient_x = eval_context.target_energy.energy_and_grad(x_batch)
        energies.append(energy.to(DEVICE))
        scores.append(-(gradient_x.to(DEVICE).reshape(len(y_batch), -1) @ Q))
    return torch.cat(energies), torch.cat(scores)

def drift_and_divergence(y: torch.Tensor, t: torch.Tensor):
    velocity_parts, divergence_parts = [], []
    for start in range(0, len(y), DRIFT_BATCH):
        leaf = y[start : start + DRIFT_BATCH].detach().requires_grad_(True)
        with torch.enable_grad():
            velocity = drift(leaf, t)
            divergence = torch.zeros(len(leaf), device=DEVICE)
            for _ in range(HUTCHINSON_SAMPLES):
                probe = torch.empty_like(leaf).bernoulli_(0.5).mul_(2).sub_(1)
                vector_jacobian = torch.autograd.grad(
                    (velocity * probe).sum(), leaf, retain_graph=True
                )[0]
                divergence += (vector_jacobian * probe).sum(1) / HUTCHINSON_SAMPLES
        velocity_parts.append(velocity.detach())
        divergence_parts.append(divergence.detach())
    return torch.cat(velocity_parts), torch.cat(divergence_parts)

def normalized_ess(logw: torch.Tensor) -> float:
    weights = torch.softmax(logw, dim=0)
    return float(1 / (len(weights) * weights.square().sum()))

def run_prop3(resample: bool = True, seed: int = SEED + 2):
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    y, ancestry = sample_gmm(NUM_PARTICLES, generator=generator, return_component=True)
    logw = torch.zeros(NUM_PARTICLES, device=DEVICE)
    dt = 1.0 / NUM_ANNEALING_STEPS
    diagnostics = {
        "t": [], "ess": [], "resampled": [],
        "raw_mode_ancestry": [], "weighted_mode_ancestry": [],
    }
    num_resamples = 0

    for step in range(NUM_ANNEALING_STEPS):
        t = torch.tensor((step + 0.5) * dt, device=DEVICE)
        U0, score0 = gmm_energy_score(y)
        U1, score1 = target_energy_score(y)
        velocity, divergence = drift_and_divergence(y, t)
        grad_Ut = -((1 - t) * score0 + t * score1)

        logw += (divergence - (grad_Ut * velocity).sum(1) + U0 - U1) * dt
        y += (-EPSILON * grad_Ut + velocity) * dt
        y += math.sqrt(2 * EPSILON * dt) * torch.randn(y.shape, device=DEVICE, generator=generator)

        current_ess = normalized_ess(logw)
        normalized_weights = torch.softmax(logw, dim=0)
        weighted_mode_ancestry = torch.stack([
            normalized_weights[ancestry == mode].sum()
            for mode in range(NUM_COMPONENTS)
        ])
        did_resample = bool(resample and current_ess < ESS_THRESHOLD)
        if did_resample:
            index = torch.multinomial(torch.softmax(logw, 0), len(logw), replacement=True, generator=generator)
            y = y[index]
            ancestry = ancestry[index]
            logw.zero_()
            num_resamples += 1

        diagnostics["t"].append(float((step + 1) * dt))
        diagnostics["ess"].append(current_ess)
        diagnostics["resampled"].append(did_resample)
        diagnostics["raw_mode_ancestry"].append(
            (torch.bincount(ancestry, minlength=NUM_COMPONENTS).float() / len(ancestry)).cpu()
        )
        diagnostics["weighted_mode_ancestry"].append(
            weighted_mode_ancestry.cpu()
        )
        if not torch.isfinite(y).all() or not torch.isfinite(logw).all():
            raise FloatingPointError(f"Non-finite Proposition-3 state at step {step + 1}")
        print(
            f"step {step + 1:3d}/{NUM_ANNEALING_STEPS}: "
            f"ESS/N={current_ess:.4f}, resampled={did_resample}"
        )

    target_energy, _ = target_energy_score(y)
    return {
        "y": y.detach(),
        "samples": y_to_x(y).detach(),
        "target_energy": target_energy.detach(),
        "logw": logw.detach(),
        "diagnostics": diagnostics,
        "num_resamples": num_resamples,
    }


In [ ]:
method2 = run_prop3(resample=True)
print({
    "final_segment_ESS/N": normalized_ess(method2["logw"]),
    "resampling_events": method2["num_resamples"],
    "particles": len(method2["samples"]),
})


## 7. Matched evaluation

The repository evaluator computes the same energy Wasserstein, Ramachandran-torus Wasserstein, TICA Wasserstein, and clustering metrics used by the paper. If the official SBG artifact exists, both methods are evaluated together against the same test reference.


In [ ]:
def samples_data_to_cpu(data: SamplesData) -> SamplesData:
    return SamplesData(
        samples=data.samples.detach().cpu(),
        E_target=data.E_target.detach().cpu(),
        logw=data.logw.detach().cpu() if data.logw is not None else None,
    )

comparison_samples = {
    "gmm_prop3": SamplesData(
        method2["samples"].cpu(),
        method2["target_energy"].cpu(),
        logw=method2["logw"].cpu(),
    )
}
sbg_diagnostics = None
if SBG_SAMPLE_FILE.exists():
    official = torch.load(SBG_SAMPLE_FILE, map_location="cpu", weights_only=False)
    comparison_samples["sbg_smc"] = samples_data_to_cpu(official["smc"])
    if SBG_DIAGNOSTICS_FILE.exists():
        sbg_diagnostics = torch.load(SBG_DIAGNOSTICS_FILE, map_location="cpu", weights_only=False)
else:
    print("Official SBG artifact not found; evaluating Method 2 only.")

evaluator = PeptideEnsembleEvaluator(
    fix_symmetry=True,
    drop_unfixable_symmetry=False,
    num_eval_samples=min(10_000, NUM_PARTICLES),
    do_plots=False,
)
metrics = evaluator.evaluate(comparison_samples, eval_context, prefix="test/Ace-A-Nme")
for key, value in sorted(metrics.items()):
    if any(token in key for token in ("wasserstein", "effective-sample-size", "median-energy", "mean-energy", "jsd")):
        scalar = float(value) if isinstance(value, (float, int, torch.Tensor)) else value
        print(f"{key}: {scalar}")


## 8. Ramachandran, ESS, and mode-ancestry diagnostics

The SBG SMC output is resampled at the endpoint, so its output weights are uniform. Its pre-resampling ESS trajectory is read from the saved diagnostics. Method-2 ESS is the segment ESS between resampling events; resampling times are marked explicitly. For the mode-informed prior, raw ancestry fractions after resampling and weighted ancestry mass immediately before resampling are plotted separately. These are initial-mode ancestries, not a reclassification of the evolving coordinates.


In [ ]:
plot_sets = [("test MD", eval_context.true_data.samples, None, False)]
if "sbg_smc" in comparison_samples:
    plot_sets.append(("official SBG SMC", comparison_samples["sbg_smc"].samples, None, True))
plot_sets.append(("mode-informed GMM + Prop. 3", comparison_samples["gmm_prop3"].samples, comparison_samples["gmm_prop3"].logw, True))

fig, axes = plt.subplots(1, len(plot_sets), figsize=(5 * len(plot_sets), 4.2), constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, (title, samples, logw, normalized) in zip(axes, plot_sets):
    phi, psi = rama(samples, normalized=normalized)
    if logw is None:
        ax.hexbin(phi, psi, gridsize=55, bins="log", mincnt=1)
    else:
        weights = torch.softmax(logw, 0).numpy()
        ax.hexbin(phi, psi, C=weights, reduce_C_function=np.sum, gridsize=55, mincnt=1)
    ax.set(xlim=(-180, 180), ylim=(-180, 180), xlabel=r"$\phi$ (deg)", ylabel=r"$\psi$ (deg)", title=title)
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
times = np.asarray(method2["diagnostics"]["t"])
method2_ess = np.asarray(method2["diagnostics"]["ess"])
ax.plot(times, method2_ess, label="Method 2: Prop. 3")
for time, flag in zip(times, method2["diagnostics"]["resampled"]):
    if flag:
        ax.axvline(time, color="C0", alpha=0.25, linestyle="--")
if sbg_diagnostics is not None:
    sbg_diag = sbg_diagnostics["diagnostics"]
    ax.plot(np.asarray(sbg_diag["t"], float), np.asarray(sbg_diag["ess"], float), label="Official SBG SMC")
ax.axhline(ESS_THRESHOLD, color="black", linestyle=":", label="resampling threshold")
ax.set(xlabel="annealing time", ylabel="ESS/N", ylim=(0, 1), title="Annealing weight efficiency")
ax.legend()
plt.show()

raw_mode_ancestry = np.stack(
    method2["diagnostics"]["raw_mode_ancestry"]
)
weighted_mode_ancestry = np.stack(
    method2["diagnostics"]["weighted_mode_ancestry"]
)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for mode in range(NUM_COMPONENTS):
    axes[0].plot(
        times, raw_mode_ancestry[:, mode], label=f"mode {mode}"
    )
    axes[1].plot(
        times, weighted_mode_ancestry[:, mode], label=f"mode {mode}"
    )
axes[0].set(
    xlabel="annealing time", ylabel="particle fraction", ylim=(0, 1),
    title="Raw initial-mode ancestry after resampling",
)
axes[1].set(
    xlabel="annealing time", ylabel="weighted fraction", ylim=(0, 1),
    title="Weighted initial-mode ancestry before resampling",
)
for ax in axes:
    ax.legend()
plt.show()


## Interpretation checklist

Use the comparison for three separate questions:

1. **Proposal/transport quality:** compare raw Ramachandran and target-energy distributions before final weighting or resampling.
2. **Correction efficiency:** compare ESS trajectories and the number/timing of resampling events. Do not interpret the final uniform SMC weights as an ESS of the original trajectories.
3. **Final equilibrium quality:** compare energy, torus, and TICA Wasserstein metrics after resampling.

For a paper-matched run, use `SMOKE_TEST=False`, run the official baseline once with `RUN_OFFICIAL_SBG=True`, and repeat both methods over seeds 0, 1, and 2. The reported SBG baseline applies energy and log-weight filtering; Method 2 is left unfiltered by default so its Proposition-3 correction remains transparent. If filtering is introduced, report filtered and unfiltered results separately.
